# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 2 — Refresh / Content Opportunity Scoring. Task type: ranking / scoring.**

Not classification. A classifier answers "is this page declining, yes or no?" for every page, but nobody consumes that. An editor has a fixed number of review hours, so the real question is "which pages first?" — and `framing-ml-problems` maps that straight to ranking, scored with precision@K.

The distinction is practical, not academic. Framed as classification I would chase accuracy across all 30,000 pages, most of which nobody will ever open. Framed as ranking, I am judged on the top of the list, which is the only part anyone acts on.

Mechanically I still fit a classifier and sort by its predicted probability — that is how the ranking gets produced. But the task is ranking, because the ordering is the deliverable.

**The action the output supports.** The ranked queue goes to a content/SEO editor, who takes the top N their week allows and picks one action per page: **refresh, expand, protect, prune, or monitor** — each with a reason code, so they can disagree with the score.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label = (trend_direction == "down")` — a binary column I derive myself, then rank by its predicted probability.

**Where it comes from — and why it is a proxy, not the outcome.** The outcome I actually care about is "would refreshing this page pay off?" Nobody labelled that; it would need an experiment where someone refreshed a random half of these pages. What I have instead is `trend_direction`, a bucket already computed in the shipped data from `trend_pct`.

So the honest description is: **observed, but pre-bucketed, and one step removed from the decision.** It is measured from real traffic rather than invented by a rule, which is what `framing-ml-problems` asks for. But it is a *recorded trend bucket*, not future traffic and not refresh ROI. Every claim I make inherits that gap, and I would rather write it here than have a reader assume otherwise.

**The leakage consequence.** Because the label is derived from `trend_direction`, which comes from `trend_pct`, neither column can ever be a feature — they are the answer in disguise. Notebook 02 showed what happens if they slip in: the tree splits on `trend_pct` and scores near-perfectly while learning nothing.

**What would make it better.** The warehouse daily table (`fact_content_daily_performance`) has real dates, so in Week 3 I can define a forward-looking window label — "did traffic fall over the *next* 30 days?" — and check whether this framing survives a target that is genuinely in the future.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.** Of the top 50 pages my ranking flags, what fraction are actually declining?

**Why 50, and not 20 or 500.** It is one editor-week of review capacity. Tying K to what a person can actually do keeps the metric attached to the decision — a metric measured at a depth nobody reviews is a metric nobody should care about.

**What "good" means, stated before I train anything:**

| Bar | Value | What it means |
|---|---|---|
| Random ordering (holdout base rate) | **0.391** | The floor. Below this, shuffling the list would be better. |
| Shipped hand rule | **0.240** | The current transparent baseline — already below random. |
| My target | **beat 0.391 first, then 0.240 comfortably** | Clearing the hand rule is not enough; it loses to random. |

So "good" is not a single number I pick after the fact. It is: **clear the base rate, then beat the hand rule by more than the noise.**

**How I will report it.** Averaged over repeated client-holdout splits, quoted with its spread — never a single split. In Week 2 the same comparison moved by 0.14 between splits, which was enough to reverse which method looked better. One split is not a result.

**The guardrail.** Precision@50 only grades the 50 pages I surfaced; it is silent about the ones I missed. Since a missed declining page is the more expensive error, I will also track recall among high-impression declining pages so the metric cannot flatter me by ranking a safe, easy 50.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os
from pathlib import Path
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ONE ROW = ONE PAGE (content item), measured over a trailing 90-day window.
# Grain probe: if content_id is unique, the row really is the unit of analysis.
print(f"rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,} "
      f"| one row = one page? {df['content_id'].is_unique}")

# My lane's slice: pages visible enough that a refresh could plausibly matter.
# (Reviewing a page nobody sees is not a content decision worth an editor's hour.)
lane = df[df["impressions_90d"] >= 500].copy()
print(f"lane slice (impressions_90d >= 500): {len(lane):,} pages across "
      f"{lane['client_id'].nunique()} clients\n")

# Sketch the target column: derived here, never taken as a feature.
lane["is_declining_label"] = lane["trend_direction"].str.lower().eq("down").astype(int)

print("Target column sketch — is_declining_label:")
print(lane["is_declining_label"].value_counts().rename({0: "not declining", 1: "declining"}).to_string())
print(f"positive rate: {lane['is_declining_label'].mean():.3f}\n")

# The unit of analysis as an actual dataframe: a few pre-decision signals,
# the pseudonymous grouping key, and the target. No URLs, titles, or queries exist
# in this dataset at all -- content_id and client_id are pseudonyms used for
# grouping and splitting only, never as features.
cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "avg_position", "ctr", "word_count", "is_declining_label"]
lane[cols].head(5)


rows: 30,000 | unique content_id: 30,000 | one row = one page? True
lane slice (impressions_90d >= 500): 16,726 pages across 28 clients

Target column sketch — is_declining_label:
is_declining_label
declining        9961
not declining    6765
positive rate: 0.596



,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I checked this rather than asserting it, because "use ML" is the assumption most worth attacking.

**1. The fixed rule already exists, and it loses to random.** `scripts/02_baseline_score.py` is a sensible hand rule — stale AND visible, ranked by impressions. Measured on a client holdout it scores **0.240 Precision@50 against a 0.391 base rate**. Someone already wrote the if-statement, and shuffling the queue would beat it.

**2. No single signal carries the answer.** In the trained model, the strongest feature reaches only **0.161** importance, and it takes **5 features to account for half** of the total across 52. A rule readable by a human can hold maybe three or four conditions. The signal is spread thinner than that — which is the specific shape `framing-ml-problems` describes as ML earning its place: the pattern is real but too tangled to hand-write.

**3. The thresholds cannot be global.** Per-client decline rates run from **0.00 to 0.94**. Any fixed cutoff like "180 days stale" is simultaneously too strict for one client and too loose for another. A model can weight signals differently per page; a single hand-written threshold cannot.

**What would change my mind.** This is an argument for a *learned* ranking, not proof one is required. In ML-07 I will rebuild the hand rule **per client** rather than globally. If that alone clears 0.391 and lands near the model, the honest answer is to ship the rule — it is readable, free to run, and an editor can argue with it. ML has to keep earning its place, not win once and stay.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
